# Domain 1: Agentic Architecture & Orchestration
## TS 1.3: Configure Subagent Invocation, Context Passing, and Spawning
**Certification Focus**: CCAR-F Domain 1 (27% Exam Weight) | **Scenario Alignment**: Scenario 3 (Multi-Agent Research System)

---

### 🎯 Key Architectural Principles
1. **The `Task` Tool**: The primary mechanism for spawning subagents. Must be explicitly declared in `tools` or `allowedTools`.
2. **Parallel Subagent Spawning**: Emitting multiple `Task` tool use blocks in a **single coordinator turn/response** enables concurrent subagent execution, reducing latency by 40%–60%.
3. **Structured Attribution & Metadata Separation**: Content and findings must be separated from tracking metadata (`source_url`, `timestamp`, `page_number`, `confidence`).
4. **Goal-Oriented Instructions**: Prompt subagents with explicit deliverables, schemas, and constraints rather than rigid step-by-step procedures.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.3 Subagent Invocation & Spawning.")

### 📋 Step 2: Agent Definitions & The Task Tool Schema
The `Task` tool declaration allows Claude to invoke registered subagents dynamically with dedicated prompts and metadata.

In [ ]:
# Step 2: Subagent Registry & Task Tool Schema
@dataclass
class AgentDefinition:
    name: str
    description: str
    system_prompt: str
    allowed_tools: List[str]

SUBAGENT_REGISTRY = {
    "web_researcher": AgentDefinition(
        name="web_researcher",
        description="Researches technical documentation and articles.",
        system_prompt="You are a precise technical researcher.",
        allowed_tools=["fetch_url", "search"],
    ),
    "code_analyzer": AgentDefinition(
        name="code_analyzer",
        description="Inspects source code repositories, classes, and syntax.",
        system_prompt="You are an expert static code analyst.",
        allowed_tools=["grep", "read_file"],
    ),
}

TASK_TOOL = {
    "name": "Task",
    "description": (
        "Spawns an isolated subagent to perform a targeted task. "
        "Multiple Task calls may be emitted in a single turn for concurrent execution."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subagent_type": {
                "type": "string",
                "enum": ["web_researcher", "code_analyzer"],
                "description": "Registered subagent type to instantiate.",
            },
            "prompt": {
                "type": "string",
                "description": "Complete, self-contained prompt for the subagent.",
            },
            "task_metadata": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string"},
                    "expected_output_format": {"type": "string"},
                },
                "required": ["task_id"],
            },
        },
        "required": ["subagent_type", "prompt", "task_metadata"],
    },
}

print(" Task Tool schema and subagent registry defined.")

### ⚡ Step 3: Parallel Execution Engine
Simulates executing multiple subagent tasks spawned concurrently by the coordinator.

In [ ]:
# Step 3: Subagent Execution Engine
def execute_subagent_task(subagent_type: str, prompt: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an individual subagent task with metadata/content separation."""
    if subagent_type not in SUBAGENT_REGISTRY:
        return {"status": "error", "error": f"Unknown subagent: {subagent_type}"}

    if subagent_type == "web_researcher":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Claude Agent SDK uses the Task tool for spawning child agents.",
                    "metadata": {
                        "source_url": "https://docs.anthropic.com/agents",
                        "confidence": 0.98,
                    },
                }
            ],
        }
    elif subagent_type == "code_analyzer":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Found 2 occurrences of allowedTools containing 'Task'.",
                    "metadata": {
                        "file": "src/coordinator.ts",
                        "lines": [42, 88],
                    },
                }
            ],
        }
    return {"status": "error", "error": "Unhandled type"}

def run_parallel_spawning_coordinator(user_query: str, client: Any) -> Dict[str, Any]:
    """Simulates coordinator receiving a prompt and issuing parallel Task calls."""
    # Turn 1: Model emits parallel tool calls
    resp = client.messages.create(
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": user_query}],
        tools=[TASK_TOOL],
    )

    completed_tasks = []
    if resp.stop_reason == "tool_use":
        for block in resp.content:
            if block.type == "tool_use" and block.name == "Task":
                result = execute_subagent_task(
                    block.input["subagent_type"],
                    block.input["prompt"],
                    block.input["task_metadata"],
                )
                completed_tasks.append(result)

    return {"spawned_count": len(completed_tasks), "tasks": completed_tasks}

print(" Parallel subagent dispatcher ready.")

### 🧪 Step 4: Offline Simulation Walkthrough
We configure a mock response emitting **two** `Task` calls in a single turn to demonstrate parallel subagent spawning.

In [ ]:
# Step 4: Parallel Spawning Simulation
# Mock coordinator issuing TWO concurrent Task calls in Turn 1
parallel_resp = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_web_01",
            input_data={
                "subagent_type": "web_researcher",
                "prompt": "Search for Agent SDK Task tool guidelines.",
                "task_metadata": {"task_id": "T-WEB-001"},
            },
        ),
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_code_02",
            input_data={
                "subagent_type": "code_analyzer",
                "prompt": "Locate allowedTools definitions in repository.",
                "task_metadata": {"task_id": "T-CODE-002"},
            },
        ),
    ],
    stop_reason="tool_use",
)

mock_client = MockClaudeClient([parallel_resp])
sim_res = run_parallel_spawning_coordinator("Analyze Task tool usage", mock_client)

print(f"Spawned Subagent Tasks: {sim_res['spawned_count']}")
for t in sim_res["tasks"]:
    print(f"  - Task ID: {t['task_id']}, Type: {t['subagent_type']}, Status: {t['status']}")

### 🔬 Step 5: Verification Assertions
We assert that:
1. Two distinct subagents were invoked in parallel within the same turn.
2. Output payloads cleanly separate substantive content from tracking metadata.

In [ ]:
# Step 5: Verification Assertions
assert sim_res["spawned_count"] == 2, "Expected 2 parallel subagent spawns"

web_task = next(t for t in sim_res["tasks"] if t["subagent_type"] == "web_researcher")
assert web_task["task_id"] == "T-WEB-001"
finding = web_task["findings"][0]
assert "content" in finding
assert "metadata" in finding
assert "source_url" in finding["metadata"]

print(" All verification assertions passed! Parallel spawning verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Spawning subagents requires `Task` in `allowedTools`.
- **Rule 2**: Emitting multiple `Task` calls in a single turn achieves true concurrency (reducing latency 40-60%).
- **Rule 3**: Separate content from metadata (`source_url`, `confidence`, `timestamp`) to preserve provenance.